# Phenikaa Campus Courier — PIPELINE SUY LUẬN HOÀN CHỈNH (không train)

**Luồng:** ảnh test + `observations.json` → resize 1024 → **YOLOv8m** → đồ thị → **BamiBERT** đọc câu nhiệm vụ → giải `goal_ref/via_ref` → **planner 10 robot** → bước đi đầu tiên → `submission.json`

| Khối | Lấy nguyên văn từ | Ghi chú |
|---|---|---|
| NLU | `nlu_bamibert_phase3.ipynb` | model 8 head + `decode()` có ràng buộc + `parse_missions`. `USE_TTA=False` (đúng kết quả đo trên val). Ckpt: ft3 → ft2 → best (cái nào có trước) |
| YOLO → graph | `YoloToGraph_YOLOv8m.ipynb` | `predict_dets` + `dets_to_scene` bản cuối (300/300 val), `conf=0.4`, `split_long=True`. Ảnh **resize 1024×1024 INTER_AREA → lưu .jpg** y hệt lúc build dataset train |
| Robot | `Untitled24.ipynb` | `F3_WEIGHTS` + `calculate_cost` + Dijkstra/greedy bản có `round(..., 9)` (đạt 100% train) |

**Mã hành động:** `0 = UP, 1 = DOWN, 2 = LEFT, 3 = RIGHT`, `-1` = không có đường / không có robot. Mỗi scene xuất 10 số (robot 0 → 9).

**Phần MỚI duy nhất (không có trong 3 notebook):** glue code — đọc `observations.json`, resize ảnh test, *giải `goal_ref`/`via_ref` thành toạ độ node* (near/far/bắc/nam/đông/tây), ghi file nộp.

## 1. Cài đặt + mount Drive + giải nén dữ liệu

In [1]:
!pip install -q ultralytics "transformers<=5.5.0" sentencepiece protobuf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 141.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 10.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o "/content/drive/MyDrive/Phenikaa_Campus_Courier_2026.zip" -d "/content/dataset"

Mounted at /content/drive


## 2. Import + cấu hình

In [3]:
import os, warnings
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'; os.environ['YOLO_VERBOSE'] = 'False'; os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
warnings.filterwarnings('ignore')
import re, json, glob, math, heapq, shutil, unicodedata, collections
from pathlib import Path
import cv2
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from functools import partial
from tqdm.auto import tqdm
tqdm = partial(tqdm, disable=True)      # tắt thanh tiến trình
from transformers import AutoTokenizer, AutoModel
import transformers; transformers.logging.set_verbosity_error()

# ---------------- ĐƯỜNG DẪN ----------------
ROOT      = "/content/dataset/Phenikaa_Campus_Courier_2026/delivery_public"
TEST_DIR  = f"{ROOT}/test"
VAL_DIR   = f"{ROOT}/validation"

YOLO_PATH = "/content/drive/MyDrive/YOLO_Courier_Backup_M/best.pt"
NLU_CKPT  = "/content/drive/MyDrive/bamibert_nlu_ft711.pt"
RESIZED_DIR     = "/content/test_1024"       # ảnh test sau khi resize 1024
OUT_SUBMISSION  = "/content/submission.json"
OUT_FLAT        = "/content/submission_flat.json"      # bản dự phòng: list phẳng 10 số / scene (giống labels.json)
DRIVE_OUT_DIR   = "/content/drive/MyDrive"             # copy kết quả lên Drive (lỗi thì bỏ qua)

# ---------------- THAM SỐ ĐÃ CHỐT (đúng như notebook gốc) ----------------
IMGSZ         = 1024      # kích thước lúc train YOLO
YOLO_MIN_CONF = 0.1       # collect_dets(min_conf=0.1) rồi lọc lại bằng conf=0.4 (đường chạy đạt 300/300 val)
YOLO_CONF     = 0.5
MODEL_NAME    = "Qualcomm-AI-Research/BamiBERT"
MAX_LEN       = 192
USE_TTA       = False     # đo trên val: TTA không giúp -> tắt (cell suy luận NLU gốc in ra USE_TTA = False)

# ---------------- KHOÁ TRONG observations.json (None = tự dò) ----------------
OBS_ID_KEY = None         # vd 'scene_id' / 'id'
OBS_IMAGE_KEY = None      # vd 'image'
OBS_TEXT_KEY = None       # vd 'mission_text' / 'text'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == 'cuda'
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16


## 3. Resize TẤT CẢ ảnh về 1024×1024 (làm trước, chưa cần nạp model)

In [4]:
# =====================================================================
# RESIZE 1024  (y hệt build_split lúc train: cv2.imread -> resize INTER_AREA -> lưu .jpg)
# =====================================================================
def resize_to_1024(src_path, dst_path):
    img = cv2.imread(src_path)
    if img is None:
        raise FileNotFoundError(f"Không đọc được ảnh: {src_path}")
    cv2.imwrite(dst_path, cv2.resize(img, (IMGSZ, IMGSZ), interpolation=cv2.INTER_AREA))
    return img.shape[:2]


# =====================================================================
# ĐỌC observations.json (tự dò khoá id / ảnh / câu nhiệm vụ)
# =====================================================================
_ID_KEYS   = ('scene_id', 'id', 'obs_id', 'observation_id', 'sample_id', 'uid', 'name')
_IMG_KEYS  = ('image', 'image_path', 'img', 'img_path', 'file', 'filename', 'file_name', 'path')
_TEXT_KEYS = ('mission_text', 'text', 'instruction', 'command', 'mission', 'nlu_text', 'query', 'prompt', 'description')
_IMG_EXT   = ('.png', '.jpg', '.jpeg', '.bmp', '.webp')

def _as_records(data):
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for k in ('observations', 'data', 'items', 'samples', 'scenes', 'records'):
            if isinstance(data.get(k), list):
                return data[k]
        return [({**v, '_key': k} if isinstance(v, dict) else {'_key': k, 'text': v}) for k, v in data.items()]
    raise ValueError("observations.json có cấu trúc lạ (không phải list/dict)")

def _first_str(d, keys):
    for k in keys:
        v = d.get(k)
        if isinstance(v, str) and v.strip():
            return v, k
        if isinstance(v, dict):                                   # vd "mission": {"text": "..."}
            for kk in ('text', 'mission_text', 'instruction'):
                if isinstance(v.get(kk), str) and v[kk].strip():
                    return v[kk], f"{k}.{kk}"
    return None, None

def _get_key(d, key):                                              # hỗ trợ khoá lồng "mission.text"
    for part in key.split('.'):
        d = d[part]
    return d

def extract_fields(o, i):
    # --- ảnh
    if OBS_IMAGE_KEY:
        img = str(_get_key(o, OBS_IMAGE_KEY))
    else:
        img, _ = _first_str(o, _IMG_KEYS)
        if img is None or not img.lower().endswith(_IMG_EXT):
            img = next((v for v in o.values() if isinstance(v, str) and v.lower().endswith(_IMG_EXT)), img)
    # --- câu nhiệm vụ
    if OBS_TEXT_KEY:
        text = str(_get_key(o, OBS_TEXT_KEY))
    else:
        text, _ = _first_str(o, _TEXT_KEYS)
        if text is None:                                           # fallback: chuỗi dài nhất có khoảng trắng, không phải đường dẫn ảnh
            cands = [v for v in o.values() if isinstance(v, str) and ' ' in v and not v.lower().endswith(_IMG_EXT)]
            text = max(cands, key=len) if cands else None
    # --- id
    if OBS_ID_KEY:
        sid = str(_get_key(o, OBS_ID_KEY))
    else:
        sid, _ = next(((str(o[k]), k) for k in _ID_KEYS if isinstance(o.get(k), (str, int)) and not isinstance(o.get(k), bool)),
                      (None, None))
        if sid is None:
            sid = str(o.get('_key') or (os.path.splitext(os.path.basename(img))[0] if img else i))
    if img is None:
        raise KeyError(f"Không tìm thấy trường ảnh trong observation #{i}: khoá có = {list(o.keys())}. Hãy đặt OBS_IMAGE_KEY.")
    if text is None:
        raise KeyError(f"Không tìm thấy câu nhiệm vụ trong observation #{i}: khoá có = {list(o.keys())}. Hãy đặt OBS_TEXT_KEY.")
    return sid, img, text

def resolve_image(split_dir, ref):
    stem = os.path.splitext(os.path.basename(ref))[0]
    for p in (os.path.join(split_dir, ref), os.path.join(split_dir, 'images', os.path.basename(ref))):
        if os.path.isfile(p):
            return p
    hits = sorted(glob.glob(os.path.join(split_dir, 'images', stem + '.*')))
    if hits:
        return hits[0]
    raise FileNotFoundError(f"Không thấy ảnh '{ref}' trong {split_dir}")

# observations.json có 1 dòng / (ảnh, robot): vd "validation-00000-R0" ... "-R9" -> cùng 1 ảnh + cùng 1 câu nhiệm vụ.
# => GỘP theo ảnh: YOLO / NLU / dựng đồ thị chỉ chạy 1 LẦN / ảnh; 10 robot chỉ khác nhau ở id (vòng for r in range(10) trong first_steps).
_ROBOT_SUFFIX = re.compile(r'[-_ ]?R(?:obot)?[-_ ]?(\d+)$', re.I)

def load_records(split_dir, limit=None):
    # KHÔNG lưu câu nhiệm vụ vào record: extract_fields trả text thì vứt ngay (chỉ để kiểm tra có trường text).
    data = json.load(open(os.path.join(split_dir, 'observations.json'), encoding='utf-8'))
    obs = _as_records(data)
    groups = collections.OrderedDict()                       # basename ảnh -> record (giữ thứ tự xuất hiện đầu tiên)
    for i, o in enumerate(obs):
        sid, img, _ = extract_fields(o, i)
        key = os.path.basename(str(img))
        g = groups.get(key)
        if g is None:
            groups[key] = {'scene_id': _ROBOT_SUFFIX.sub('', sid), 'image': img, 'obs_ids': [sid]}
        else:
            g['obs_ids'].append(sid)
    del data, obs
    recs = []
    for g in groups.values():
        if limit and len(recs) >= limit: break               # limit = số ẢNH (scene), không phải số dòng
        g['idx'] = len(recs)
        g['img_path'] = resolve_image(split_dir, g['image'])
        recs.append(g)
    return recs


def run_nlu(split_dir, recs, bs=128):
    # NƠI DUY NHẤT câu nhiệm vụ tồn tại: đọc -> đưa thẳng vào NLU -> biến mất khi hàm return.
    # Không gán vào record, không print, không ghi file. Trả về list kết quả NLU (đã parse) theo thứ tự recs.
    data = json.load(open(os.path.join(split_dir, 'observations.json'), encoding='utf-8'))
    first = {}                                               # basename ảnh -> câu đầu tiên (như bản cũ)
    for i, o in enumerate(_as_records(data)):
        _, img, text = extract_fields(o, i)
        first.setdefault(os.path.basename(str(img)), text)
    del data
    keys = [os.path.basename(str(r['image'])) for r in recs]
    parsed = []
    for s in tqdm(range(0, len(keys), bs), desc="NLU"):
        parsed += parse_missions([first[k] for k in keys[s:s + bs]], tta=USE_TTA)
    first.clear()
    return parsed


def resize_all(split_dir, resized_dir, limit=None):
    # đọc observations.json và RESIZE TẤT CẢ ảnh về 1024x1024 (lưu .jpg) trước khi chạy bất kỳ model nào
    os.makedirs(resized_dir, exist_ok=True)
    recs = load_records(split_dir, limit)
    for r in tqdm(recs, desc=f"Resize 1024 -> {resized_dir}"):
        r['resized'] = os.path.join(resized_dir, f"{r['idx']:05d}_{os.path.splitext(os.path.basename(r['img_path']))[0]}.jpg")
        r['orig_hw'] = resize_to_1024(r['img_path'], r['resized'])
    return recs

test_records = resize_all(TEST_DIR, RESIZED_DIR)       # <- resize toàn bộ ảnh test ngay tại đây

## 4. NLU — model BamiBERT 8 head + nạp checkpoint

In [5]:
PLACES = ['canteen','clinic','dorm','gate','lab','lecture','library','office','parking','sports']
KINDS  = ['none','near','far','north','south','east','west']
HEADS = {'goal': PLACES, 'via': ['none']+PLACES,
         'gkind': KINDS, 'ganchor': ['none']+PLACES,     # goal_ref.kind / .anchor
         'vkind': KINDS, 'vanchor': ['none']+PLACES,     # via_ref.kind / .anchor
         'urgent': ['no','yes'], 'fragile': ['no','yes']}
HEAD_W = {'goal':2.0,'via':2.0,'gkind':1.0,'ganchor':1.0,'vkind':1.0,'vanchor':1.0,'urgent':1.2,'fragile':1.0}
H2I = {h:{l:i for i,l in enumerate(ls)} for h,ls in HEADS.items()}

def strip_accents(t):
    t = t.replace('đ','d').replace('Đ','D')
    return ''.join(c for c in unicodedata.normalize('NFD', t) if unicodedata.category(c) != 'Mn')
def fold(s): return strip_accents(s).lower()

CORE_HEADS = ('goal', 'via', 'urgent', 'fragile')
NP_ = len(PLACES); NF_IDX = [H2I['gkind']['near'], H2I['gkind']['far']]

def decode(P, rules=None):
    # P: dict head -> xác suất (N, C). Chọn tổ hợp nhãn HỢP LỆ có tổng log-xác suất lớn nhất.
    rules = RULES if rules is None else rules
    L = {h: np.log(np.clip(np.asarray(P[h], dtype=np.float64), 1e-9, 1.0)) for h in HEADS}
    N = len(L['goal']); out = {h: np.zeros(N, dtype=int) for h in HEADS}
    out['urgent'], out['fragile'] = L['urgent'].argmax(1), L['fragile'].argmax(1)

    def best_ref(lk, la, own):
        best = (-1e18, 0, 0)
        for k in range(len(KINDS)):
            if rules['anchor_iff']:
                if k in NF_IDX:
                    cands = [a for a in range(1, NP_ + 1) if not (rules['anchor_ne_own'] and a == own)]
                    a = max(cands, key=lambda j: la[j])
                else: a = 0
            else: a = int(la.argmax())
            s = lk[k] + la[a]
            if s > best[0]: best = (s, k, a)
        return best

    for i in range(N):
        G = [best_ref(L['gkind'][i], L['ganchor'][i], g + 1) for g in range(NP_)]
        V = [(L['vkind'][i][0] + L['vanchor'][i][0], 0, 0) if rules['via_none_clears']
             else best_ref(L['vkind'][i], L['vanchor'][i], 0)]
        V += [best_ref(L['vkind'][i], L['vanchor'][i], v) for v in range(1, NP_ + 1)]
        best = (-1e18, 0, 0)
        for g in range(NP_):
            for v in range(NP_ + 1):
                if rules['goal_ne_via'] and v == g + 1: continue
                s = L['goal'][i][g] + L['via'][i][v] + G[g][0] + V[v][0]
                if s > best[0]: best = (s, g, v)
        _, g, v = best
        out['goal'][i], out['via'][i] = g, v
        out['gkind'][i], out['ganchor'][i] = G[g][1], G[g][2]
        out['vkind'][i], out['vanchor'][i] = V[v][1], V[v][2]
    return out

# ràng buộc mặc định (sẽ được ghi đè bằng RULES lưu trong checkpoint ở cell tiếp theo)
RULES = {'goal_ne_via': True, 'anchor_iff': True, 'anchor_ne_own': True, 'via_none_clears': True}

In [6]:
class MultiHeadNLU(nn.Module):
    def __init__(self, name, extra_pools=3):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        hidden = self.enc.config.hidden_size
        self.extra_pools = extra_pools
        self.drop = nn.Dropout(0.2)
        self.pool_q = nn.ModuleDict({
            head: nn.Sequential(nn.Linear(hidden, hidden // 2), nn.Tanh(), nn.Linear(hidden // 2, 1))
            for head in HEADS
        })
        self.heads = nn.ModuleDict({head: nn.Linear(hidden * 2, len(labels)) for head, labels in HEADS.items()})
        self.extra_pool_q = nn.ModuleDict({head: nn.Linear(hidden, extra_pools, bias=False) for head in HEADS})
        extra_size = hidden * (extra_pools + 1)
        self.extra_norm = nn.ModuleDict({head: nn.LayerNorm(extra_size) for head in HEADS})
        self.extra_heads = nn.ModuleDict({head: nn.Linear(extra_size, len(labels)) for head, labels in HEADS.items()})
        for head in HEADS:
            nn.init.zeros_(self.pool_q[head][2].weight)
            nn.init.zeros_(self.pool_q[head][2].bias)
            nn.init.normal_(self.extra_pool_q[head].weight, mean=0.0, std=0.02)
            nn.init.zeros_(self.extra_heads[head].weight)
            nn.init.zeros_(self.extra_heads[head].bias)

    def forward(self, ids, mask, return_emb=False):
        hidden = self.enc(input_ids=ids, attention_mask=mask).last_hidden_state
        cls = hidden[:, 0, :]; logits = {}; token_mask = mask.unsqueeze(-1) == 0
        for head in HEADS:
            weight = self.pool_q[head](hidden).squeeze(-1).float().masked_fill(mask == 0, -1e4).softmax(-1).to(hidden.dtype)
            pooled = (weight.unsqueeze(-1) * hidden).sum(1)
            old_logits = self.heads[head](self.drop(torch.cat([cls, pooled], -1)))

            extra_score = self.extra_pool_q[head](hidden).float().masked_fill(token_mask, -1e4)
            extra_weight = extra_score.softmax(dim=1).to(hidden.dtype)
            extra_pool = torch.einsum('btk,bth->bkh', extra_weight, hidden).flatten(1)
            extra_feature = self.extra_norm[head](torch.cat([cls, extra_pool], -1))
            logits[head] = old_logits + self.extra_heads[head](self.drop(extra_feature))

        if not return_emb: return logits
        mask3 = mask.unsqueeze(-1).to(hidden.dtype)
        mean = (hidden * mask3).sum(1) / mask3.sum(1).clamp(min=1)
        emb = F.normalize(F.normalize(cls.float(), dim=-1) + F.normalize(mean.float(), dim=-1), dim=-1)
        return logits, emb

@torch.no_grad()
def predict_probs(model, texts, bs=64):
    model.eval(); texts = list(texts)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i])); chunks = {h: [] for h in HEADS}
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok([texts[i] for i in idx], padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(device)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            lg = model(enc['input_ids'], enc['attention_mask'])
        for h in HEADS: chunks[h].append(lg[h].float().softmax(-1).cpu().numpy())
    inv = np.empty(len(order), dtype=int); inv[np.array(order)] = np.arange(len(order))
    return {h: np.concatenate(chunks[h])[inv] for h in HEADS}

_BRACKET = re.compile(r'\[[^\]]*\]')
def tta_views(t):
    base = re.sub(r'\s+', ' ', _BRACKET.sub(' ', t)).strip() or t      # bỏ cụm [Đơn #1234]
    return [t, base, strip_accents(base)]
TTA_W = (2.0, 1.0, 1.0)
def predict_probs_tta(model, texts):
    acc = None
    for vi, w in enumerate(TTA_W):
        P = predict_probs(model, [tta_views(t)[vi] for t in texts])
        acc = {h: w * P[h] for h in HEADS} if acc is None else {h: acc[h] + w * P[h] for h in HEADS}
    return {h: v / sum(TTA_W) for h, v in acc.items()}


def parse_missions(texts, tta=None):
    tta = USE_TTA if tta is None else tta
    P = predict_probs_tta(model, texts) if tta else predict_probs(model, texts); pred = decode(P); out = []
    for i, t in enumerate(texts):
        p = {h: HEADS[h][int(pred[h][i])] for h in HEADS}
        ref = lambda k, a: None if p[k] == 'none' else {'kind': p[k], 'anchor': None if p[a] == 'none' else p[a]}
        out.append({'goal': p['goal'], 'goal_ref': ref('gkind', 'ganchor'),
                    'via': None if p['via'] == 'none' else p['via'],
                    'via_ref': ref('vkind', 'vanchor') if p['via'] != 'none' else None,
                    'urgent': p['urgent'] == 'yes', 'fragile': p['fragile'] == 'yes'})
    return out


# ---------------- nạp tokenizer + checkpoint ----------------
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

assert os.path.exists(NLU_CKPT), f"Không thấy checkpoint NLU: {NLU_CKPT} — sửa NLU_CKPT ở cell cấu hình"

_ck = torch.load(NLU_CKPT, map_location=device)
model = MultiHeadNLU(MODEL_NAME).to(device)            # tên `model` giữ nguyên vì parse_missions() gọi biến global này
model.load_state_dict(_ck['state']); model.eval()
RULES = _ck.get('rules', RULES)
del _ck

## 5. YOLO → đồ thị scene (nguyên văn từ notebook YOLO)

In [7]:
CLASSES = [
    'canteen', 'clinic', 'dorm', 'gate', 'lab', 'lecture', 'library', 'office', 'parking', 'sports',
    'robot_up', 'robot_down', 'robot_left', 'robot_right',
    'edge_type_1', 'edge_type_2', 'edge_type_3',
    'print_edge_1', 'print_edge_2', 'print_edge_3',
    'closed', 'stairs', 'oneway_up', 'oneway_down', 'oneway_left', 'oneway_right',
    'weather_sun', 'weather_cloud', 'weather_rain',
    'swatch_edge_1', 'swatch_edge_2', 'swatch_edge_3',
    'swatch_print_1', 'swatch_print_2', 'swatch_print_3',
    'txt_normal', 'txt_crowded', 'txt_covered'
]

EDGE_CLS = ('edge_type_', 'print_edge_')                  # box cạnh (kiểu nét) -> status lấy từ legend
EDGE_BOX_CLS = {'closed'}                                 # closed = box phủ cả đoạn đường
MODIFIER_CLS = {'stairs', 'oneway_up', 'oneway_down', 'oneway_left', 'oneway_right'}  # icon nhỏ giữa cạnh
NODE_CLS = set(CLASSES[:14])                              # landmark + robot


# ==========================================
# HELPERS
# ==========================================
def get_center(box):
    return ((box[0] + box[2]) / 2, (box[1] + box[3]) / 2)


def get_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def dynamic_cluster(coords, unit_size):
    if not coords:
        return []
    coords = sorted(coords)
    clusters = [[coords[0]]]
    tolerance = unit_size * 0.4
    for c in coords[1:]:
        if c - np.mean(clusters[-1]) <= tolerance:
            clusters[-1].append(c)
        else:
            clusters.append([c])
    return [float(np.mean(cl)) for cl in clusters]


def get_grid_index(val, clusters):
    return min(range(len(clusters)), key=lambda i: abs(clusters[i] - val))


def is_plain_edge(c):
    return c.startswith(EDGE_CLS)


# ==========================================
# LEGEND: swatch k -> txt bên cạnh -> status
# ==========================================
def build_legend(dets, txt_side='right'):
    texts = [d for d in dets if d['class'].startswith('txt_')]
    swatches = [d for d in dets if d['class'].startswith(('swatch_edge_', 'swatch_print_'))]
    if not texts or not swatches:
        return {}

    def find_label(s):
        sx1, _, sx2, _ = s['box']
        slack = 0.3 * s['w']
        row = [t for t in texts
               if abs(t['center'][1] - s['center'][1]) <= max(s['h'], t['h']) * 0.6]
        if txt_side == 'left':
            side = [t for t in row if t['box'][2] <= sx1 + slack]
        else:
            side = [t for t in row if t['box'][0] >= sx2 - slack]
        pool = side or row or texts
        return min(pool, key=lambda t: get_distance(t['center'], s['center']))

    best = {}  # k -> (conf swatch, status)
    for s in swatches:
        k = s['class'][-1]
        t = find_label(s)
        status = t['class'].replace('txt_', '')
        if k not in best or s['conf'] > best[k][0]:
            best[k] = (s['conf'], status)

    legend = {}
    for k, (_, status) in best.items():
        legend['edge_type_' + k] = status
        legend['print_edge_' + k] = status
    return legend


def _oneway_target(direction, na, nb):
    if direction == 'up':
        return [min(na[0], nb[0]), na[1]]
    if direction == 'down':
        return [max(na[0], nb[0]), na[1]]
    if direction == 'left':
        return [na[0], min(na[1], nb[1])]
    return [na[0], max(na[1], nb[1])]  # right


# ==========================================
# 1. YOLO -> DETS (chạy model 1 lần, lọc ngưỡng sau)
# ==========================================
def predict_dets(model, image_path, min_conf=0.1):
    res = model.predict(image_path, conf=min_conf, verbose=False)[0]
    img = res.orig_img[:, :, ::-1]  # BGR -> RGB
    dets = []
    for box, cls_id, cf in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.cls.cpu().numpy(),
                               res.boxes.conf.cpu().numpy()):
        box = [float(v) for v in box]
        dets.append({'class': CLASSES[int(cls_id)], 'box': box, 'conf': float(cf),
                     'center': get_center(box), 'w': box[2] - box[0], 'h': box[3] - box[1]})
    return dets, img


# ==========================================
# 2. DETS -> JSON GRAPH
#    box cạnh   : edge_type_k / print_edge_k (status = legend) / closed
#    icon nhỏ   : stairs / oneway_*  -> gán vào cạnh có box chứa tâm icon
#    split_long : box cạnh dài >1 ô -> tách thành nhiều cạnh 1 ô (False = giữ kiểu cũ: 1 cạnh nối xa)
# ==========================================
def dets_to_scene(dets, mission_via=None, mission_goal=None, conf=0.4, edge_conf=None,
                  txt_side='right', split_long=True):
    edge_conf = conf if edge_conf is None else edge_conf
    dets = [dict(d) for d in dets
            if d['conf'] >= (edge_conf if is_plain_edge(d['class']) else conf)]

    legend = build_legend(dets, txt_side=txt_side)
    for d in dets:
        if is_plain_edge(d['class']):
            d['status'] = legend.get(d['class'], '?')

    edge_dets = [d for d in dets if is_plain_edge(d['class']) or d['class'] in EDGE_BOX_CLS]
    modifiers = [d for d in dets if d['class'] in MODIFIER_CLS]

    # thước đo ô lưới: chỉ đo trên box cạnh (không tính icon)
    hw = [d['w'] for d in edge_dets if d['w'] > d['h']]
    vh = [d['h'] for d in edge_dets if d['h'] > d['w']]
    grid_w = float(np.median(hw)) if hw else 100.0
    grid_h = float(np.median(vh)) if vh else 100.0

    scene = {'nodes': [], 'edges': [], 'landmarks': [], 'robot': {},
             'weather': 'rain' if any(d['class'] == 'weather_rain' for d in dets) else 'dry',
             'mission': {'via': mission_via, 'goal': mission_goal}}

    # --- lưới động
    raw_x, raw_y = [], []
    for d in dets:
        if d['class'] in NODE_CLS:
            raw_x.append(d['center'][0])
            raw_y.append(d['center'][1])
    for d in edge_dets:
        x1, y1, x2, y2 = d['box']
        if d['w'] >= d['h']:
            raw_y.append(d['center'][1])
            raw_x += [x1, x2]
        else:
            raw_x.append(d['center'][0])
            raw_y += [y1, y2]
    col_cl, row_cl = dynamic_cluster(raw_x, grid_w), dynamic_cluster(raw_y, grid_h)

    # --- landmark / robot
    for d in dets:
        if d['class'] not in NODE_CLS:
            continue
        r, c = get_grid_index(d['center'][1], row_cl), get_grid_index(d['center'][0], col_cl)
        if d['class'].startswith('robot_'):
            scene['robot'] = {'rc': [r, c], 'heading': d['class'].split('_')[1].upper()}
        else:
            scene['landmarks'].append({'type': d['class'], 'rc': [r, c]})
    for r in range(len(row_cl)):
        for c in range(len(col_cl)):
            scene['nodes'].append({'rc': [r, c]})

    # --- cạnh
    edges = {}
    for d in edge_dets:
        x1, y1, x2, y2 = d['box']
        horiz = d['w'] >= d['h']
        if horiz:
            r = get_grid_index(d['center'][1], row_cl)
            ca, cb = sorted([get_grid_index(x1, col_cl), get_grid_index(x2, col_cl)])
            steps = [([r, c], [r, c + 1]) for c in range(ca, cb)]
        else:
            c = get_grid_index(d['center'][0], col_cl)
            ra, rb = sorted([get_grid_index(y1, row_cl), get_grid_index(y2, row_cl)])
            steps = [([r, c], [r + 1, c]) for r in range(ra, rb)]
        if not steps:
            continue
        if len(steps) > 1 and not split_long:
            steps = [(steps[0][0], steps[-1][1])]

        for na, nb in steps:
            if horiz:
                box = [col_cl[na[1]], y1, col_cl[nb[1]], y2]
            else:
                box = [x1, row_cl[na[0]], x2, row_cl[nb[0]]]
            key = (tuple(na), tuple(nb))
            e = edges.setdefault(key, {'a': na, 'b': nb, 'status': None, 'stairs': False,
                                       'oneway_to': None, '_box': box, '_horiz': horiz,
                                       '_center': get_center(box), '_plain_conf': -1.0,
                                       '_closed': False})
            cls = d['class']
            if is_plain_edge(cls):
                if d['conf'] > e['_plain_conf']:      # 2 box plain chồng nhau -> lấy box conf cao hơn
                    e['_plain_conf'] = d['conf']
                    e['status'] = legend.get(cls, 'normal')
            else:  # closed
                e['_closed'] = True

    for e in edges.values():
        if e['_closed']:
            e['status'] = 'closed'

    # --- icon nhỏ -> cạnh có box chứa tâm icon, đúng hướng
    tol = 0.15 * min(grid_w, grid_h)
    # --- icon nhỏ -> cạnh có diện tích đè (overlap) lớn nhất với icon
    for m in modifiers:
        mx1, my1, mx2, my2 = m['box']
        m_area = (mx2 - mx1) * (my2 - my1)
        direction = m['class'].split('_')[1] if m['class'].startswith('oneway_') else None

        best, max_overlap_area = None, 0.0

        for e in edges.values():
            ex1, ey1, ex2, ey2 = e['_box']

            # 1. Kiểm tra hướng tương thích trước khi xét diện tích đè
            if direction in ('left', 'right') and not e['_horiz']:
                continue
            if direction in ('up', 'down') and e['_horiz']:
                continue

            # 2. Tính diện tích phần giao nhau (Intersection) giữa box icon và box cạnh
            inter_x1 = max(mx1, ex1)
            inter_y1 = max(my1, ey1)
            inter_x2 = min(mx2, ex2)
            inter_y2 = min(my2, ey2)

            if inter_x2 > inter_x1 and inter_y2 > inter_y1:
                overlap_area = (inter_x2 - inter_x1) * (inter_y2 - inter_y1)

                # Chọn cạnh có diện tích đè lớn nhất
                if overlap_area > max_overlap_area:
                    max_overlap_area = overlap_area
                    best = e

        # Nếu không có cạnh nào đè lên icon (diện tích giao = 0)
        if best is None or max_overlap_area == 0:
            continue

        if m['class'] == 'stairs':
            best['stairs'] = True
        else:
            best['oneway_to'] = _oneway_target(direction, best['a'], best['b'])

    for e in edges.values():
        for k in ('_box', '_horiz', '_center', '_plain_conf', '_closed'):
            e.pop(k, None)
        scene['edges'].append(e)
    return scene

In [8]:
from ultralytics import YOLO
yolo = YOLO(YOLO_PATH)
assert int(yolo.overrides.get('imgsz', IMGSZ)) == IMGSZ, "imgsz của ckpt khác 1024 — kiểm tra lại model"

## 6. Robot — chi phí + Dijkstra/greedy (nguyên văn từ notebook robot)

In [9]:
# ----- cell 3 gốc: bảng trọng số + calculate_cost -----
F3_WEIGHTS = {
    0: {"all": {"crowded_dry": 0.0, "crowded_rain": 0.0, "covered_dry": 0.0, "covered_rain": 0.0, "right": 0.0, "left": 0.0, "uturn": 0.0, "oneway": 0.0, "oneway_crowded": 0.0, "stairs_dry": 0.0, "stairs_rain": 0.0}},
    1: {"all": {"crowded_dry": 5.0, "crowded_rain": 5.0, "covered_dry": 0.0, "covered_rain": 0.0, "right": 0.0, "left": 0.0, "uturn": 0.0, "oneway": 0.0, "oneway_crowded": 0.0, "stairs_dry": 0.0, "stairs_rain": 0.0}},
    2: {"all": {"crowded_dry": 0.0, "crowded_rain": 0.0, "covered_dry": -0.5, "covered_rain": -0.5, "right": 0.0, "left": 0.0, "uturn": 0.0, "oneway": 0.0, "oneway_crowded": 0.0, "stairs_dry": 0.0, "stairs_rain": 0.0}},
    3: {"all": {"crowded_dry": 2.0, "crowded_rain": 0.0, "covered_dry": 0.0, "covered_rain": -0.8, "right": 0.0, "left": 0.0, "uturn": 0.0, "oneway": 0.0, "oneway_crowded": 0.0, "stairs_dry": 0.0, "stairs_rain": 0.0}},
    4: {"all": {"crowded_dry": 1.0, "crowded_rain": 1.0, "covered_dry": 0.0, "covered_rain": 0.0, "right": 0.0, "left": 0.0, "uturn": 0.0, "oneway": 0.0, "oneway_crowded": 0.0, "stairs_dry": 0.0, "stairs_rain": 4.0}},
    5: {"all": {"crowded_dry": 0.0, "crowded_rain": 0.0, "covered_dry": 0.0, "covered_rain": 0.0, "right": 3.0, "left": 3.0, "uturn": 31.0, "oneway": 0.0, "oneway_crowded": 0.0, "stairs_dry": 0.0, "stairs_rain": 0.0}},
    6: {"urgent": {"crowded_dry": 0.0, "crowded_rain": 0.0, "covered_dry": 0.0, "covered_rain": 0.0, "right": 0.0, "left": 0.0, "uturn": 0.0, "oneway": 0.0, "oneway_crowded": 0.0, "stairs_dry": 0.0, "stairs_rain": 0.0}, "calm": {"crowded_dry": 4.0, "crowded_rain": 4.0, "covered_dry": -0.25, "covered_rain": -0.25, "right": 0.0, "left": 0.0, "uturn": 0.0, "oneway": 0.0, "oneway_crowded": 0.0, "stairs_dry": 0.0, "stairs_rain": 0.0}},
    7: {"fragile": {"crowded_dry": 6.0, "crowded_rain": 6.0, "covered_dry": 0.0, "covered_rain": 0.0, "right": 1.5, "left": 1.5, "uturn": 15.0, "oneway": 0.0, "oneway_crowded": 0.0, "stairs_dry": 0.0, "stairs_rain": 0.0}, "normal": {"crowded_dry": 0.0, "crowded_rain": 0.0, "covered_dry": 0.0, "covered_rain": 0.0, "right": 0.0, "left": 0.0, "uturn": 0.0, "oneway": 0.0, "oneway_crowded": 0.0, "stairs_dry": 0.0, "stairs_rain": 0.0}},
    8: {"all": {"crowded_dry": 0.0, "crowded_rain": 0.0, "covered_dry": 0.0, "covered_rain": 0.0, "right": 0.5, "left": 4.5, "uturn": 9.0, "oneway": 0.0, "oneway_crowded": 0.0, "stairs_dry": 0.0, "stairs_rain": 0.0}},
    9: {"all": {"crowded_dry": 2.0, "crowded_rain": 2.0, "covered_dry": 0.0, "covered_rain": 0.0, "right": 0.0, "left": 0.0, "uturn": 0.0, "oneway": 0.0, "oneway_crowded": 0.0, "stairs_dry": 0.0, "stairs_rain": 0.0}},
}

def calculate_cost(robot_id, edge_data, curr_heading, next_heading, mission, weather):
    rain = (weather == 'rain')
    status = edge_data.get('status', 'normal')
    turn = (next_heading - curr_heading) % 4          # 0 thẳng, 1 phải, 2 quay đầu, 3 trái

    if robot_id == 6:
        ctx = 'urgent' if mission.get('urgent', False) else 'calm'
    elif robot_id == 7:
        ctx = 'fragile' if mission.get('fragile', False) else 'normal'
    else:
        ctx = 'all'

    w = F3_WEIGHTS.get(robot_id, {}).get(ctx)
    cost = 1.0

    if w is None:
        return cost

    if status == 'crowded':
        cost += w['crowded_rain'] if rain else w['crowded_dry']
    elif status == 'covered':
        cost += w['covered_rain'] if rain else w['covered_dry']

    if turn == 1: cost += w['right']
    elif turn == 3: cost += w['left']
    elif turn == 2: cost += w['uturn']

    if edge_data.get('oneway_to') is not None:
        cost += w['oneway']
        if status == 'crowded': cost += w['oneway_crowded']

    if edge_data.get('stairs', False):
        cost += w['stairs_rain'] if rain else w['stairs_dry']

    return cost


# ----- cell 4 gốc: planner (R9 = greedy, R0-R8 = Dijkstra theo (node, heading)) -----
DIR_MAPPING = {(-1, 0): 0, (0, 1): 1, (1, 0): 2, (0, -1): 3}
STR_TO_DIR = {"UP": 0, "RIGHT": 1, "DOWN": 2, "LEFT": 3}
DIR_TO_SUBMISSION = {0: 0, 2: 1, 3: 2, 1: 3}
TIE_BREAKER = {"STRAIGHT": 0, "RIGHT_TURN": 1, "LEFT_TURN": 2, "U_TURN": 3}

def get_turn_type(curr_heading, next_heading):
    turn_val = (next_heading - curr_heading) % 4
    if turn_val == 0: return "STRAIGHT"
    if turn_val == 1: return "RIGHT_TURN"
    if turn_val == 2: return "U_TURN"
    if turn_val == 3: return "LEFT_TURN"

def build_adj_list(scene, robot_id):
    G = {}
    for edge in scene['edges']:
        if edge['status'] == 'closed': continue
        if edge.get('stairs', False) and robot_id != 4: continue

        u, v = tuple(edge['a']), tuple(edge['b'])
        if edge.get('oneway_to') is None:
            G.setdefault(u, []).append((v, edge))
            G.setdefault(v, []).append((u, edge))
        else:
            oneway_dest = tuple(edge['oneway_to'])
            if oneway_dest == v: G.setdefault(u, []).append((v, edge))
            elif oneway_dest == u: G.setdefault(v, []).append((u, edge))
    return G

def get_nodes_by_type(scene, node_type, ref=None):
    if not node_type: return []
    if ref and ref.get('rc'):
        return [tuple(ref['rc'])]
    targets = []
    for lm in scene['landmarks']:
        if lm['type'] == node_type:
            targets.append(tuple(lm['rc']))
    return targets

def greedy_step(scene, robot_id):
    G = build_adj_list(scene, robot_id)
    start_node = tuple(scene['robot']['rc'])
    start_heading = STR_TO_DIR[scene['robot']['heading']]

    if start_node not in G:
        return -1

    mission = scene.get('mission', {})
    vias = get_nodes_by_type(scene, mission.get('via'), mission.get('via_ref'))
    goals = get_nodes_by_type(scene, mission.get('goal'), mission.get('goal_ref'))
    targets = vias if vias else goals

    # Nếu không có mục tiêu (targets rỗng do NLU/YOLO lỗi), thay vì crash,
    # ta chọn tạm các neighbor xung quanh làm mục tiêu giả để R9 vẫn đi được!
    if not targets:
        targets = [neighbor for neighbor, _ in G.get(start_node, [])]

    if start_node in targets:
        # Nếu đang đứng ở đích, nếu vẫn còn đường khác thì đi, không thì thôi
        other_cands = [n for n, _ in G.get(start_node, []) if n != start_node]
        if not other_cands:
            return -1
        targets = other_cands

    def dist_manhattan(n, g): return abs(n[0]-g[0]) + abs(n[1]-g[1])

    # Lúc này targets chắc chắn không rỗng, hàm min() sẽ không bao giờ bị lỗi ValueError nữa
    best_target = min(targets, key=lambda g: dist_manhattan(start_node, g))

    cands = []
    for neighbor, edge_data in G.get(start_node, []):
        dr = neighbor[0] - start_node[0]
        dc = neighbor[1] - start_node[1]
        next_heading = DIR_MAPPING.get((dr, dc))
        if next_heading is None: continue

        # Dùng id 9 cho robot R9
        step_cost = calculate_cost(9, edge_data, start_heading, next_heading, mission, scene.get('weather', 'dry'))

        d = dist_manhattan(neighbor, best_target)
        turn_type = get_turn_type(start_heading, next_heading)
        tie = (TIE_BREAKER[turn_type],)

        total_score = step_cost + (1024.0 * d)
        cands.append((total_score, tie, next_heading, neighbor))

    if not cands: return -1
    cands.sort()
    raw_dir = cands[0][2]
    return DIR_TO_SUBMISSION[raw_dir]

def get_all_shortest_paths(G, start_node, start_heading, robot_id, mission, weather):
    pq = [(0, (), start_node, start_heading, None)]
    min_costs = {(start_node, start_heading): (0, (), None)}

    while pq:
        cost, tie_score, curr_node, curr_heading, first_move_node = heapq.heappop(pq)

        known_cost, known_tie, _ = min_costs[(curr_node, curr_heading)]
        if cost > known_cost or (cost == known_cost and tie_score > known_tie):
            continue

        for neighbor, edge_data in G.get(curr_node, []):
            dr = neighbor[0] - curr_node[0]
            dc = neighbor[1] - curr_node[1]
            next_heading = DIR_MAPPING.get((dr, dc))
            if next_heading is None: continue

            step_cost = calculate_cost(robot_id, edge_data, curr_heading, next_heading, mission, weather)
            new_cost = round(cost + step_cost, 9)
            new_first_move = first_move_node if first_move_node is not None else neighbor
            turn_type = get_turn_type(curr_heading, next_heading)
            new_tie_score = tie_score + (TIE_BREAKER[turn_type],)

            current = min_costs.get((neighbor, next_heading))
            if current is None or new_cost < current[0] or (new_cost == current[0] and new_tie_score < current[1]):
                min_costs[(neighbor, next_heading)] = (new_cost, new_tie_score, new_first_move)
                heapq.heappush(pq, (new_cost, new_tie_score, neighbor, next_heading, new_first_move))

    return min_costs

# def simulate_robot_dijkstra(scene, robot_id):
#     if robot_id == 9: return greedy_step(scene, robot_id)

#     G = build_adj_list(scene, robot_id)
#     start_node = tuple(scene['robot']['rc'])

#     if start_node not in G:
#         return -1

#     start_heading = STR_TO_DIR[scene['robot']['heading']]
#     mission = scene['mission']
#     weather = scene.get('weather', 'dry')

#     vias = get_nodes_by_type(scene, mission.get('via'), mission.get('via_ref'))
#     goals = get_nodes_by_type(scene, mission.get('goal'), mission.get('goal_ref'))

#     best_overall_first_move = None
#     start_costs = get_all_shortest_paths(G, start_node, start_heading, robot_id, mission, weather)

#     if vias:
#         best_overall = None
#         via_cache = {}

#         for via in vias:
#             for via_heading in range(4):
#                 if (via, via_heading) not in start_costs: continue
#                 c1, tie1, first_move = start_costs[(via, via_heading)]

#                 if (via, via_heading) not in via_cache:
#                     via_cache[(via, via_heading)] = get_all_shortest_paths(G, via, via_heading, robot_id, mission, weather)

#                 via_costs = via_cache[(via, via_heading)]

#                 best_goal_cost, best_goal_tie = float('inf'), ()
#                 for goal in goals:
#                     for goal_heading in range(4):
#                         if (goal, goal_heading) not in via_costs: continue
#                         c2, tie2, _ = via_costs[(goal, goal_heading)]
#                         if c2 < best_goal_cost or (c2 == best_goal_cost and tie2 < best_goal_tie):
#                             best_goal_cost = c2
#                             best_goal_tie = tie2

#                 if best_goal_cost != float('inf'):
#                     total_cost = round(c1 + best_goal_cost, 9)
#                     total_tie = tie1 + best_goal_tie
#                     if best_overall is None or total_cost < best_overall[0] or (total_cost == best_overall[0] and total_tie < best_overall[1]):
#                         best_overall = (total_cost, total_tie, first_move)

#         if best_overall:
#             best_overall_first_move = best_overall[2]

#     else:
#         best_goal = None
#         for goal in goals:
#             for goal_heading in range(4):
#                 if (goal, goal_heading) not in start_costs: continue
#                 c, tie, first_move = start_costs[(goal, goal_heading)]
#                 if best_goal is None or c < best_goal[0] or (c == best_goal[0] and tie < best_goal[1]):
#                     best_goal = (c, tie, first_move)
#         if best_goal:
#             best_overall_first_move = best_goal[2]

#     if best_overall_first_move is None: return -1
#     dr = best_overall_first_move[0] - start_node[0]
#     dc = best_overall_first_move[1] - start_node[1]
#     return DIR_TO_SUBMISSION[DIR_MAPPING[(dr, dc)]]


def simulate_robot_dijkstra(scene, robot_id):
    if robot_id == 9: return greedy_step(scene, robot_id)

    G = build_adj_list(scene, robot_id)
    if 'robot' not in scene or not scene['robot']: return -1

    start_node = tuple(scene['robot']['rc'])
    if start_node not in G: return -1

    start_heading = STR_TO_DIR.get(scene['robot'].get('heading', 'UP'), 0)
    mission = scene['mission']
    weather = scene.get('weather', 'dry')

    vias = get_nodes_by_type(scene, mission.get('via'), mission.get('via_ref'))
    goals = get_nodes_by_type(scene, mission.get('goal'), mission.get('goal_ref'))

    best_overall_first_move = None
    start_costs = get_all_shortest_paths(G, start_node, start_heading, robot_id, mission, weather)

    if vias:
        best_overall = None
        via_cache = {}

        for via in vias:
            for via_heading in range(4):
                if (via, via_heading) not in start_costs: continue
                c1, tie1, first_move = start_costs[(via, via_heading)]

                ck = (via, via_heading)
                if ck not in via_cache:
                    via_costs = get_all_shortest_paths(G, via, via_heading, robot_id, mission, weather)
                    best_goal_cost, best_goal_tie = float('inf'), ()
                    for goal in goals:
                        for goal_heading in range(4):
                            if (goal, goal_heading) not in via_costs: continue
                            c2, tie2, _ = via_costs[(goal, goal_heading)]
                            if c2 < best_goal_cost or (c2 == best_goal_cost and tie2 < best_goal_tie):
                                best_goal_cost = c2
                                best_goal_tie = tie2
                    via_cache[ck] = (best_goal_cost, best_goal_tie)

                best_goal_cost, best_goal_tie = via_cache[ck]

                if best_goal_cost != float('inf'):
                    total_cost = round(c1 + best_goal_cost, 9)
                    total_tie = tie1 + best_goal_tie
                    if best_overall is None or total_cost < best_overall[0] or (total_cost == best_overall[0] and total_tie < best_overall[1]):
                        best_overall = (total_cost, total_tie, first_move)

        if best_overall:
            best_overall_first_move = best_overall[2]

    else:
        best_goal = None
        for goal in goals:
            for goal_heading in range(4):
                if (goal, goal_heading) not in start_costs: continue
                c, tie, first_move = start_costs[(goal, goal_heading)]
                if best_goal is None or c < best_goal[0] or (c == best_goal[0] and tie < best_goal[1]):
                    best_goal = (c, tie, first_move)
        if best_goal: best_overall_first_move = best_goal[2]

    # =========================================================================
    # 💡 CƠ CHẾ FALLBACK THÔNG MINH KHI KHÔNG TÌM THẤY ĐƯỜNG ĐI ĐẾN ĐÍCH
    # =========================================================================
    if best_overall_first_move is None:
        neighbors = G.get(start_node, [])
        if neighbors:
            # Lấy toàn bộ tọa độ các landmark mà YOLO nhìn thấy trong ảnh
            all_landmarks = [tuple(lm['rc']) for lm in scene.get('landmarks', [])]

            if all_landmarks:
                # Chọn ô hàng xóm có khoảng cách gần nhất tới bất kỳ landmark nào
                def dist_to_landmarks(n):
                    return min(abs(n[0] - lm[0]) + abs(n[1] - lm[1]) for lm in all_landmarks)
                best_neighbor = min(neighbors, key=lambda x: dist_to_landmarks(x[0]))[0]
            else:
                # Nếu bản đồ không có landmark nào, chọn ô hàng xóm hợp lệ đầu tiên
                best_neighbor = neighbors[0][0]

            best_overall_first_move = best_neighbor
        else:
            return -1  # Hoàn toàn bị kẹt (không có cạnh nào thông ra ngoài)

    # Chuyển đổi bước đi đầu tiên thành action submission (0..3)
    dr = best_overall_first_move[0] - start_node[0]
    dc = best_overall_first_move[1] - start_node[1]

    dir_key = (dr, dc)
    if dir_key in DIR_MAPPING:
        return DIR_TO_SUBMISSION[DIR_MAPPING[dir_key]]
    return -1

## 7. Glue code — resize ảnh, đọc observations, giải `goal_ref`/`via_ref`, chạy 1 scene

In [10]:
# =====================================================================
# 7.1  GIẢI goal_ref / via_ref  ->  toạ độ node (rc)
#      NLU chỉ cho ra {kind, anchor}; planner (get_nodes_by_type) cần ref['rc'] khi scene có nhiều landmark cùng loại.
#      Luật (GIẢ ĐỊNH — hãy kiểm bằng cell "Kiểm tra bộ giải ref" ở cuối):
#        near/far : ứng viên gần/xa NHẤT so với landmark `anchor` (khoảng cách Manhattan trên lưới (row, col))
#        north/south : hàng nhỏ nhất / lớn nhất      east/west : cột lớn nhất / nhỏ nhất
#      Không chọn được duy nhất (hoà / thiếu anchor) -> KHÔNG gán rc -> planner xét mọi landmark cùng loại.
# =====================================================================
def _lm_nodes(scene, typ):
    return [tuple(l['rc']) for l in scene.get('landmarks', []) if l['type'] == typ]

def _unique_best(cands, key, pick_max):
    vals = [key(c) for c in cands]
    best = max(vals) if pick_max else min(vals)
    top = [c for c, v in zip(cands, vals) if v == best]
    return top[0] if len(top) == 1 else None

def resolve_ref(scene, typ, ref):
    if not typ or not ref:
        return None
    ref = {k: v for k, v in ref.items() if k != 'rc'}
    cands = _lm_nodes(scene, typ)
    if not cands:
        return ref
    if len(cands) == 1:
        ref['rc'] = list(cands[0]); return ref
    kind, anchor = ref.get('kind'), ref.get('anchor')
    pick = None

    # if kind in ('near', 'far'):
    #     anchors = _lm_nodes(scene, anchor) if anchor else []
    #     if anchors:
    #         man = lambda c: min(abs(c[0] - a[0]) + abs(c[1] - a[1]) for a in anchors)
    #         euc = lambda c: min(math.hypot(c[0] - a[0], c[1] - a[1]) for a in anchors)
    #         pick = _unique_best(cands, man, kind == 'far')
    #         if pick is None:                                     # hoà Manhattan -> thử Euclid
    #             top = [c for c in cands if man(c) == (max if kind == 'far' else min)(man(x) for x in cands)]
    #             pick = _unique_best(top, euc, kind == 'far')
    # test ưu tiên euclid trước
    if kind in ('near', 'far'):
        anchors = _lm_nodes(scene, anchor) if anchor else []
        if anchors:
            euc = lambda c: min(
                math.hypot(c[0] - a[0], c[1] - a[1])
                for a in anchors
            )
            pick = _unique_best(cands, euc, kind == 'far')

    elif kind == 'north': pick = _unique_best(cands, lambda c: c[0], False)
    elif kind == 'south': pick = _unique_best(cands, lambda c: c[0], True)
    elif kind == 'east':  pick = _unique_best(cands, lambda c: c[1], True)
    elif kind == 'west':  pick = _unique_best(cands, lambda c: c[1], False)
    if pick is not None:
        ref['rc'] = list(pick)
    return ref


# =====================================================================
# 7.2  1 scene: YOLO -> đồ thị -> mission đầy đủ -> bước đi đầu của 10 robot
# =====================================================================
def build_scene(img_path_1024, m):
    dets, _ = predict_dets(yolo, img_path_1024, min_conf=YOLO_MIN_CONF)
    scene = dets_to_scene(dets, m['via'], m['goal'], conf=YOLO_CONF, edge_conf=None, txt_side='right', split_long=True)
    scene['mission'] = {'goal': m['goal'], 'via': m['via'],
                        'goal_ref': resolve_ref(scene, m['goal'], m['goal_ref']),
                        'via_ref': resolve_ref(scene, m['via'], m['via_ref']) if m['via'] else None,
                        'urgent': m['urgent'], 'fragile': m['fragile']}
    return scene

def first_steps(scene):
    if not scene.get('robot') or 'rc' not in scene['robot']:
        return [-1] * 10
    out = []
    for r in range(10):
        try:
            out.append(int(simulate_robot_dijkstra(scene, r)))
        except Exception:
            out.append(-1)
    return out


# =====================================================================
# 7.3  PIPELINE cho cả 1 split (ảnh đã được resize sẵn ở bước trên)
# =====================================================================
def run_pipeline(split_dir, recs=None, limit=None, resized_dir=RESIZED_DIR):
    if recs is None:                                                            # chưa resize thì resize toàn bộ trước
        recs = resize_all(split_dir, resized_dir, limit)

    parsed = run_nlu(split_dir, recs)                                          # (2) NLU — text chỉ sống trong run_nlu

    for r, m in zip(tqdm(recs, desc="YOLO + graph + robot"), parsed):          # (3) YOLO -> graph -> robot
        try:
            sc = build_scene(r['resized'], m)
            r['steps'] = first_steps(sc)
        except Exception:
            r['steps'] = [-1] * 10
    return recs


# =====================================================================
# 7.4  GHI FILE NỘP — bám theo cấu trúc của sample_submission.json
# =====================================================================
_ACT_KEYS = ('action', 'step', 'move', 'direction', 'first_step', 'next_step', 'label', 'pred', 'prediction', 'value', 'dir', 'next')

def _lookup(records):
    lk = {}
    for r in records:
        b = os.path.basename(str(r['image']))
        for key in (str(r['scene_id']), str(r['image']), b, os.path.splitext(b)[0]):
            lk.setdefault(key, r['idx'])
    return lk

def build_submission(sample, records):
    acts = [r['steps'] for r in records]; N = len(records); lk = _lookup(records)
    flat = [a for row in acts for a in row]

    def idx_of(row, i):
        for v in row.values():
            if isinstance(v, (str, int)) and not isinstance(v, bool):
                for cand in (str(v), _ROBOT_SUFFIX.sub('', str(v))):      # "validation-00000-R3" -> "validation-00000"
                    if cand in lk:
                        return lk[cand]
        return i

    def robot_of(row, j):
        rk = next((k for k in row if 'robot' in str(k).lower() and isinstance(row[k], int) and not isinstance(row[k], bool)), None)
        if rk is not None: return row[rk]
        for v in row.values():                                              # robot suy từ hậu tố của id: "...-R3"
            m = isinstance(v, str) and _ROBOT_SUFFIX.search(v)
            if m: return int(m.group(1))
        return j % 10

    def fill_row(row, a):
        row = dict(row)
        for k, v in row.items():
            if isinstance(v, list) and len(v) == 10:
                row[k] = list(a); return row
            if isinstance(v, dict) and len(v) == 10:
                row[k] = {kk: a[int(re.findall(r'\d+', str(kk))[-1]) if re.findall(r'\d+', str(kk)) else j]
                          for j, kk in enumerate(v.keys())}
                return row
        rk = [k for k in row if re.fullmatch(r'(robot|bot|r)[_\- ]?\d+', str(k), re.I)]
        if len(rk) == 10:
            for k in rk: row[k] = a[int(re.findall(r'\d+', k)[-1])]
            return row
        return None

    # (1) list phẳng các số nguyên
    if isinstance(sample, list) and sample and all(isinstance(x, (int, float)) and not isinstance(x, bool) for x in sample):
        return flat, f"list phẳng {N}x10 (giống labels.json); sample có {len(sample)} phần tử"
    # (2) list các dict
    if isinstance(sample, list) and sample and isinstance(sample[0], dict):
        if len(sample) == 10 * N and len(sample) != N:                       # mỗi dòng = 1 (scene, robot)
            out = []
            for j, row in enumerate(sample):
                row = dict(row); i = idx_of(row, j // 10)
                ak = next((k for k in row if str(k).lower() in _ACT_KEYS), None)
                rid = robot_of(row, j)
                if ak is None: out = None; break
                row[ak] = acts[i][rid]; out.append(row)
            if out is not None:
                return out, "list dict, mỗi dòng = 1 (scene, robot)"
        else:
            out = []
            for i, row in enumerate(sample):
                new = fill_row(row, acts[idx_of(row, i)])
                if new is None: out = None; break
                out.append(new)
            if out is not None:
                return out, "list dict, mỗi dòng = 1 scene (10 bước đi trong 1 trường)"
    # (3) dict id -> 10 số / dict robot -> số
    if isinstance(sample, dict):
        lists = [k for k, v in sample.items() if isinstance(v, list)]
        if len(lists) == 1 and sample[lists[0]] and isinstance(sample[lists[0]][0], (dict, int)):   # {"predictions": [...]}
            inner, desc = build_submission(sample[lists[0]], records)
            return {**sample, lists[0]: inner}, f"dict bọc ngoài khoá '{lists[0]}' -> {desc}"
        if sample and all(str(k) in lk for k in sample):
            out = {}
            for k, v in sample.items():
                a = acts[lk[str(k)]]
                out[k] = ({kk: a[int(re.findall(r'\d+', str(kk))[-1]) if re.findall(r'\d+', str(kk)) else j]
                           for j, kk in enumerate(v.keys())} if isinstance(v, dict) else list(a))
            return out, "dict id -> 10 bước đi"
    # (4) không nhận ra -> dict id -> 10 số
    return {str(r['scene_id']): r['steps'] for r in records}, "KHÔNG nhận ra cấu trúc sample_submission -> dùng dict {scene_id: [10 số]} (đã có thêm submission_flat.json)"

def save_outputs(records, split_dir):
    flat = [a for r in records for a in r['steps']]
    json.dump(flat, open(OUT_FLAT, 'w'), ensure_ascii=False)
    sp = os.path.join(split_dir, 'sample_submission.json')
    sub, desc = None, 'không có sample_submission.json -> dùng list phẳng'
    if os.path.exists(sp):
        sample = json.load(open(sp, encoding='utf-8'))
        sub, desc = build_submission(sample, records)
    else:
        sub = flat
    json.dump(sub, open(OUT_SUBMISSION, 'w', encoding='utf-8'), ensure_ascii=False)


## 8. CHẠY TRÊN TẬP TEST → `submission.json`

In [11]:
# =====================================================================
# ĐẾM CÁC CA -1 (fallback)
# =====================================================================
def report_minus1(records):
    bad = [r for r in records if -1 in r['steps']]
    n_cells = sum(a == -1 for r in records for a in r['steps'])
    print(f"Tổng số ca -1: {n_cells}  |  số scene có -1: {len(bad)}/{len(records)}")

In [12]:
records = run_pipeline(TEST_DIR, recs=test_records)

save_outputs(records, TEST_DIR)                          # submission.json chỉ chứa 10 số / scene

report_minus1(records)                                   # chỉ in số ca -1


Tổng số ca -1: 0  |  số scene có -1: 0/1200
